In [1]:
#data
import pandas as pd
#plot
import matplotlib.pyplot as plt
import seaborn as sns
# interactive
from ipywidgets import interact, widgets
# sklearn
from sklearn.decomposition import PCA 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
# math
import numpy as np

import pickle


import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split

In [2]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA

class MaskedPCA(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=0.9):
        self.n_components = n_components
        self.pca = PCA(n_components=self.n_components)
        
    def fit(self, X, y=None):
        X_array = np.array(X)
        mask = ~np.isnan(X_array).any(axis=1)
        
        if np.any(mask):
            self.pca.fit(X_array[mask])
        else:
            raise ValueError("No rows without NaNs found in this column group. PCA cannot fit.")
        return self
    
    def transform(self, X):
        X_array = np.array(X)
        mask = ~np.isnan(X_array).any(axis=1)
        

        n_outputs = self.pca.n_components_
        

        output = np.full((X_array.shape[0], n_outputs), np.nan)
        
        if np.any(mask):
            output[mask] = self.pca.transform(X_array[mask])
            
        return output

In [3]:
df_iden = pd.read_csv('../../data_ieee/train_identity.csv')
df_tran = pd.read_csv('../../data_ieee/train_transaction.csv')

In [4]:
cols_V = [c for c in df_tran.columns if any(x in c[0:2] for x in ['V'])]
cols_C = sorted([c for c in df_tran.columns if any(x in c[0:2] for x in ['C'])])

df_V = df_tran[cols_V]
df_C = df_tran[cols_C]

nan_V_counts = df_V.isna().sum().sort_values(ascending=True)

In [5]:
nan_series = df_V.isna().sum()
unique_counts = nan_series.unique()
set_V = {}
for count in sorted(unique_counts):
    cols = sorted(nan_series[nan_series == count].index.tolist())
    print(f"Count V{count}: {cols[:5]}...") 
    set_V[f'Count V{count}'] = cols
set_V[f'Count C{count}'] = cols_C

Count V12: ['V279', 'V280', 'V284', 'V285', 'V286']...
Count V314: ['V100', 'V101', 'V102', 'V103', 'V104']...
Count V1269: ['V281', 'V282', 'V283', 'V288', 'V289']...
Count V76073: ['V12', 'V13', 'V14', 'V15', 'V16']...
Count V77096: ['V53', 'V54', 'V55', 'V56', 'V57']...
Count V89164: ['V75', 'V76', 'V77', 'V78', 'V79']...
Count V168969: ['V35', 'V36', 'V37', 'V38', 'V39']...
Count V279287: ['V1', 'V10', 'V11', 'V2', 'V3']...
Count V449124: ['V220', 'V221', 'V222', 'V227', 'V234']...
Count V450721: ['V169', 'V170', 'V171', 'V174', 'V175']...
Count V450909: ['V167', 'V168', 'V172', 'V173', 'V176']...
Count V460110: ['V217', 'V218', 'V219', 'V223', 'V224']...
Count V508189: ['V322', 'V323', 'V324', 'V325', 'V326']...
Count V508589: ['V143', 'V144', 'V145', 'V150', 'V151']...
Count V508595: ['V138', 'V139', 'V140', 'V141', 'V142']...


In [6]:
df = df_tran.merge(df_iden, on='TransactionID', how='left', suffixes=(None, '_new'))

df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [7]:

y = df['isFraud']
X = df.drop(columns=['isFraud', 'TransactionID'])



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cat_cols = X.select_dtypes(include=['object', 'category', 'integer']).columns.tolist()
num_cols = X.select_dtypes(include=['float']).columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")



Categorical columns: ['TransactionDT', 'ProductCD', 'card1', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
Numerical columns: ['TransactionAmt', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', '

In [ ]:

pca_parallel_steps = []
i = 1
for group_name, cols in set_V.items():
    pca_parallel_steps.append((f'pca_group_{i}', MaskedPCA(n_components=0.9), cols))
    i += 1

pca_engine = ColumnTransformer(pca_parallel_steps, remainder='passthrough')

num_pipeline = Pipeline([
    ("pca", pca_engine),
    ("imputer", SimpleImputer(strategy="constant", fill_value=0.0, add_indicator=True)), # averiguar
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

preprocesing_pipeline = preprocessor.fit(X_train)
X_processed = preprocesing_pipeline.transform(X_train)

In [9]:
X_processed

array([[-7.37061546e-02, -6.40299334e-02, -2.53111723e-01, ...,
         1.00000000e+00,  0.00000000e+00,  1.46800000e+03],
       [-7.37061611e-02, -6.40299334e-02, -2.53097342e-01, ...,
         2.00000000e+00,  1.00000000e+00,  1.63900000e+03],
       [-5.89474764e-02, -4.91966154e-02, -2.53111723e-01, ...,
         2.00000000e+00,  1.00000000e+00,  1.63900000e+03],
       ...,
       [ 6.77644388e-02,  8.57510205e-02,  1.40278479e+00, ...,
         2.00000000e+00,  1.00000000e+00,  1.63900000e+03],
       [-7.37061546e-02, -6.40299334e-02, -2.53097342e-01, ...,
         0.00000000e+00,  0.00000000e+00,  1.63900000e+03],
       [-3.60358086e-02, -6.40299334e-02, -2.53073440e-01, ...,
         2.00000000e+00,  1.00000000e+00,  1.63900000e+03]],
      shape=(472432, 203))

In [10]:

column_breakdown = {}
current_idx = 0


if "num" in preprocessor.named_transformers_:

    sample_num = preprocessor.named_transformers_['num'].transform(X[num_cols].iloc[:5])
    n_num = sample_num.shape[1]
    column_breakdown['num'] = (current_idx, current_idx + n_num)
    current_idx += n_num


if "cat" in preprocessor.named_transformers_:
    sample_cat = preprocessor.named_transformers_['cat'].transform(X[cat_cols].iloc[:5])
    n_cat = sample_cat.shape[1]
    column_breakdown['cat'] = (current_idx, current_idx + n_cat)
    current_idx += n_cat

print(f"Final Feature Map: {column_breakdown}")

Final Feature Map: {'num': (0, 170), 'cat': (170, 203)}


In [ ]:
preprocesing = {
    'preprocessor': preprocesing_pipeline,
    'num': column_breakdown['num'],
    'cat': column_breakdown['cat']
}


with open('../../models/preprocessing_config.pkl', 'wb') as f:
    pickle.dump(preprocesing, f)

with open('../../models/preprocessing_config.pkl', 'rb') as f:
    loaded_config = pickle.load(f)


pipeline = loaded_config['preprocessor']
numerical_features = loaded_config['num']
categorical_features = loaded_config['cat']



In [12]:

pd.concat([X_train, y_train], axis=1).to_csv('../../data_ieee/transactions_train.csv', index=False)
pd.concat([X_test, y_test], axis=1).to_csv('../../data_ieee/transactions_test.csv', index=False)
